In [1]:
import os, ssl

pem_path = r"C:\Users\simon.ngugi\Downloads\corp_certs.pem"
if not os.path.exists(pem_path):
    ctx = ssl.create_default_context()
    certs = ctx.get_ca_certs(binary_form=True)
    with open(pem_path, "wb") as f:
        for cert in certs:
            f.write(ssl.DER_cert_to_PEM_cert(cert).encode())

os.environ["REQUESTS_CA_BUNDLE"] = pem_path
os.environ["SSL_CERT_FILE"] = pem_path

import snowflake.connector
conn = snowflake.connector.connect(
    account="ftyokgn-cy22613", user="PHOENIX",
    password="1Leviticus.123456", database="ETL_WORKSHOP", warehouse="COMPUTE_WH",
)
cur = conn.cursor()
print("Connected")

c:\Users\simon.ngugi\Downloads\class\env\Lib\site-packages\snowflake\connector\connection\_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


Connected


In [2]:
import requests, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

all_movies = []
for page in range(1, 51):
    r = requests.get(
        "https://api.themoviedb.org/3/movie/popular",
        params={"api_key": "30df9e6cfb29fb0b594f7193054b1c1e", "page": page},
        verify=False
    )
    r.raise_for_status()
    all_movies.extend(r.json()["results"])
    if page % 10 == 0:
        print(f"  Page {page} done...")

print(f"Extracted {len(all_movies)} movies")

  Page 10 done...
  Page 20 done...
  Page 30 done...
  Page 40 done...
  Page 50 done...
Extracted 1000 movies


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

rows = []
for m in all_movies:
    rows.append({
        "movie_id":     int(m["id"]),
        "title":        m["title"],
        "release_date": m.get("release_date", ""),
        "popularity":   round(float(m["popularity"]), 2),
        "vote_average": round(float(m["vote_average"]), 2),
        "vote_count":   int(m["vote_count"]),
        "language":     m["original_language"],
        "overview":     m.get("overview", "")[:200],
        "extracted_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

df = pd.DataFrame(rows)
df.to_csv("movies_raw.csv", index=False)
print(f"Saved {len(df)} movies to movies_raw.csv")

Saved 1000 movies to movies_raw.csv


In [4]:
movies = pd.read_csv("movies_raw.csv")
movies = movies.replace({np.nan: None})

cols = ["movie_id", "title", "release_date", "popularity",
        "vote_average", "vote_count", "language", "overview", "extracted_at"]

data = [tuple(row[c] for c in cols) for _, row in movies.iterrows()]
ph = ", ".join(["%s"] * len(cols))

cur.execute("TRUNCATE TABLE BRONZE.MOVIES")
cur.executemany(f"INSERT INTO BRONZE.MOVIES ({', '.join(cols)}) VALUES ({ph})", data)
conn.commit()
print(f"Loaded {len(data)} movies into BRONZE.MOVIES")

Loaded 1000 movies into BRONZE.MOVIES


In [5]:
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Expire changed rows
cur.execute(f"""
    MERGE INTO SILVER.MOVIES_SCD2 tgt
    USING (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY movie_id ORDER BY extracted_at DESC) AS rn
            FROM BRONZE.MOVIES
        ) WHERE rn = 1
    ) src
    ON src.movie_id = tgt.movie_id AND tgt.is_current = TRUE
    WHEN MATCHED AND (
        src.popularity != tgt.popularity
        OR src.vote_average != tgt.vote_average
        OR src.vote_count != tgt.vote_count
    ) THEN UPDATE SET
        tgt.valid_to = '{now}',
        tgt.is_current = FALSE
""")

# Insert new + changed
cur.execute(f"""
    INSERT INTO SILVER.MOVIES_SCD2
    SELECT src.movie_id, src.title, src.release_date, src.popularity,
           src.vote_average, src.vote_count, src.language, src.overview,
           src.extracted_at, '{now}', NULL, TRUE
    FROM (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY movie_id ORDER BY extracted_at DESC) AS rn
            FROM BRONZE.MOVIES
        ) WHERE rn = 1
    ) src
    LEFT JOIN SILVER.MOVIES_SCD2 tgt ON src.movie_id = tgt.movie_id AND tgt.is_current = TRUE
    WHERE tgt.movie_id IS NULL
""")

conn.commit()
cur.execute("SELECT COUNT(*), SUM(CASE WHEN is_current THEN 1 ELSE 0 END) FROM SILVER.MOVIES_SCD2")
total, active = cur.fetchone()
print(f"Movies SCD2: {total} total ({active} current, {total - active} historical)")

Movies SCD2: 1087 total (992 current, 95 historical)


In [10]:
cur.close()
conn.close()
print("Done")

Done


In [7]:
# Pick 3 movies that have history
df = pd.read_sql("""
    SELECT MOVIE_ID, TITLE, POPULARITY, VOTE_AVERAGE, VOTE_COUNT,
           IS_CURRENT, VALID_FROM, VALID_TO
    FROM SILVER.MOVIES_SCD2
    WHERE MOVIE_ID IN (
        SELECT MOVIE_ID FROM SILVER.MOVIES_SCD2 WHERE IS_CURRENT = FALSE LIMIT 3
    )
    ORDER BY MOVIE_ID, VALID_FROM
""", conn)

print(f"{df['MOVIE_ID'].nunique()} movies, {len(df)} rows")
df

C:\Users\simon.ngugi\AppData\Local\Temp\ipykernel_5800\692075567.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


3 movies, 6 rows


,MOVIE_ID,TITLE,POPULARITY,VOTE_AVERAGE,VOTE_COUNT,IS_CURRENT,VALID_FROM,VALID_TO
0,10585,Child's Play,35.56,6.80,3520,False,2026-10-19 09:54:41,2026-10-19 10:01:00
1,10585,Child's Play,31.63,6.78,3520,True,2026-10-19 10:01:00,NaT
2,10625,Mean Girls,35.19,7.23,9739,False,2026-10-19 09:54:41,2026-10-19 10:01:00
3,10625,Mean Girls,31.61,7.23,9739,True,2026-10-19 10:01:00,NaT
4,137113,Edge of Tomorrow,27.96,7.64,15633,False,2026-10-19 09:54:41,2026-10-19 10:01:00
5,137113,Edge of Tomorrow,24.14,7.64,15635,True,2026-10-19 10:01:00,NaT
